# 딥러닝 모델 지도

어떤 모델이 있고, 어떤 상황에 무엇을 쓰고, 지금 흐름은 어디로 가는지.

각 절마다 PyTorch로 최소 구현을 넣어뒀어. 데이터셋 다운로드 없이 랜덤 텐서로 shape만 확인하는 구조라, 위에서부터 그냥 실행하면 끝까지 돌아가.

---

**목차**

| # | 내용 |
|---|---|
| 0 | 한 장으로 보는 지도 |
| 1 | MLP — 모든 것의 출발점 |
| 2 | CNN — 이미지의 기본기 |
| 3 | RNN / LSTM / GRU — 순서가 있는 데이터 |
| 4 | Transformer — 지금의 표준 |
| 5 | ViT — 이미지를 단어처럼 |
| 6 | 생성 모델 — AE · VAE · GAN · Diffusion · Flow Matching |
| 7 | GNN — 관계가 데이터인 경우 |
| 8 | 상황별 선택 가이드 |
| 9 | 파라미터 수 비교 |
| 10 | 2026년 현재 흐름 |

## 환경 준비

In [ ]:
import platform

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
else:
    plt.rc('font', family='NanumBarunGothic')
plt.rcParams['axes.unicode_minus'] = False

# 재현성
torch.manual_seed(2026)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch  : {torch.__version__}")
print(f"device : {device}")

In [ ]:
def count_params(model):
    '''학습 가능한 파라미터 개수를 센다.'''
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def show(name, model, x):
    '''모델에 더미 입력을 흘려보내고 입출력 shape과 파라미터 수를 출력한다.'''
    model.eval()
    with torch.no_grad():
        out = model(x)
    print(f"{name}")
    print(f"  입력   : {tuple(x.shape)}")
    print(f"  출력   : {tuple(out.shape)}")
    print(f"  파라미터: {count_params(model):,}")
    return out


# 이 노트북에서 만드는 모델들의 파라미터 수를 모아둘 곳
PARAM_TABLE = {}

---

## 0. 한 장으로 보는 지도

모델을 고르는 출발점은 성능 벤치마크가 아니라 **데이터의 모양**이야. 입력에 어떤 구조가 있는지가 아키텍처를 거의 결정해.

| 데이터의 구조 | 대표 아키텍처 | 예시 |
|---|---|---|
| 구조 없음 (표 형태, 피처 나열) | MLP, 그리고 **부스팅 트리** | 고객 이탈 예측, 신용 평가 |
| 공간적 지역성 (가까운 값끼리 관련) | CNN | 이미지, 스펙트로그램 |
| 순서 (앞뒤 맥락) | RNN/LSTM → **Transformer** | 텍스트, 시계열, 음성 |
| 집합 + 전역 관계 | Transformer | 문장, 포인트 클라우드, 멀티모달 |
| 그래프 (노드와 엣지) | GNN | 소셜 네트워크, 분자, 추천 |
| 분포 자체를 배우고 싶음 | VAE, GAN, Diffusion, Flow | 이미지·영상 생성 |

### 먼저 짚고 갈 것

**정형 데이터(표)에는 딥러닝이 답이 아닌 경우가 많아.** 행이 수만 건 수준인 CSV에서는 XGBoost·LightGBM·CatBoost가 MLP보다 거의 항상 빠르고 정확해. 전처리 손도 덜 가고, 피처 중요도 해석도 쉬워. 딥러닝이 이기기 시작하는 건 이미지·텍스트·오디오처럼 **사람이 피처를 설계하기 어려운 원시 데이터**부터야.

지금 하는 이탈 예측 같은 과제에서 로지스틱 회귀와 랜덤포레스트를 먼저 돌려보는 게 괜히 그러는 게 아니라는 뜻이야.

---

## 1. MLP (Multi-Layer Perceptron)

선형 변환과 비선형 활성화를 번갈아 쌓은 게 전부야. 모든 입력 뉴런이 모든 출력 뉴런과 연결돼서 **Fully Connected(FC)**, **Dense**라고도 불러.

$$h = \sigma(Wx + b)$$

### 강점
- 구조가 단순하고 아무 데이터에나 붙일 수 있어
- 다른 아키텍처의 마지막 분류기(head)로 거의 항상 쓰여

### 한계
- 입력의 구조를 전혀 모름. 이미지를 넣으려면 평탄화(flatten)해야 하는데, 그 순간 "옆 픽셀끼리 관련 있다"는 정보가 날아가
- 파라미터가 입력 크기에 비례해 폭증. 32×32 컬러 이미지 하나가 3072차원이라 첫 레이어만 수백만 파라미터

### 쓸 때
피처가 이미 잘 정리된 저차원 벡터일 때, 또는 다른 모델 위에 얹는 head로.

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, in_dim=3 * 32 * 32, hidden=256, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),                  # (B, 3, 32, 32) -> (B, 3072)
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.3),               # 과적합 억제
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, num_classes),
        )

    def forward(self, x):
        return self.net(x)


x_img = torch.randn(8, 3, 32, 32)          # 배치 8개의 32x32 컬러 이미지
mlp = SimpleMLP()
show("MLP", mlp, x_img)
PARAM_TABLE["MLP"] = count_params(mlp)

---

## 2. CNN (Convolutional Neural Network)

작은 필터(커널)를 이미지 위에서 슬라이딩하며 특징을 뽑아. MLP와 갈리는 지점이 두 가지야.

**지역성(locality)** — 한 번에 3×3 같은 좁은 영역만 본다. 멀리 떨어진 픽셀을 처음부터 연결하지 않아.

**파라미터 공유(weight sharing)** — 같은 필터를 이미지 전체에 재사용한다. 왼쪽 위에서 모서리를 찾는 필터가 오른쪽 아래에서도 모서리를 찾아. 그래서 파라미터가 입력 크기와 무관하고, 위치가 바뀌어도 같은 특징을 잡아내.

층이 깊어질수록 보는 범위(receptive field)가 넓어져서, 초반에는 선과 모서리, 뒤로 갈수록 질감과 물체 부분을 잡는 식으로 계층이 생겨.

### CNN 계열의 역사

| 모델 | 연도 | 핵심 아이디어 |
|---|---|---|
| LeNet-5 | 1998 | CNN의 원형. 손글씨 숫자 인식 |
| AlexNet | 2012 | ReLU + Dropout + GPU. 딥러닝 붐의 시작점 |
| VGG | 2014 | 3×3 커널만 깊게 쌓기. 단순하지만 무거움 |
| **ResNet** | 2015 | **스킵 커넥션**으로 100층 이상 학습 가능하게 만듦 |
| Inception | 2015 | 여러 크기 커널을 병렬로 |
| DenseNet | 2017 | 모든 층을 뒤쪽 층에 연결 |
| **EfficientNet** | 2019 | 깊이·너비·해상도를 **같이** 키우는 복합 스케일링 |
| **ConvNeXt** | 2022 | Transformer의 설계 기법을 CNN에 역수입 |

ResNet의 스킵 커넥션이 분기점이야. 층을 깊게 쌓으면 기울기가 사라져서 오히려 성능이 떨어지던 문제를, 입력을 출력에 그대로 더해주는 것(`out = F(x) + x`)만으로 풀었어. 지금도 거의 모든 딥러닝 아키텍처에 이 구조가 들어가 있어 — Transformer 포함.

### 쓸 때
이미지 분류·검출·분할, 그리고 오디오를 스펙트로그램으로 바꾼 경우. 데이터가 수천~수만 장 수준이면 ViT보다 CNN이 유리해.

In [ ]:
class ConvBlock(nn.Module):
    '''Conv -> BatchNorm -> ReLU. CNN의 기본 단위.'''
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)     # 학습 안정화 + 수렴 속도 향상

    def forward(self, x):
        return F.relu(self.bn(self.conv(x)))


class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 32), nn.MaxPool2d(2),    # 32x32 -> 16x16
            ConvBlock(32, 64), nn.MaxPool2d(2),   # 16x16 -> 8x8
            ConvBlock(64, 128), nn.MaxPool2d(2),  # 8x8   -> 4x4
        )
        # 전역 평균 풀링: 입력 해상도가 달라져도 동작하게 해주고 파라미터도 아낌
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)


cnn = SimpleCNN()
show("CNN", cnn, x_img)
PARAM_TABLE["CNN"] = count_params(cnn)

In [ ]:
# ResNet의 핵심인 스킵 커넥션 — 이게 전부다
class ResidualBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(ch)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(ch)

    def forward(self, x):
        identity = x                          # 입력을 따로 보관
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + identity                  # ★ 더해준다 = 스킵 커넥션
        return F.relu(out)


block = ResidualBlock(64)
feat = torch.randn(8, 64, 16, 16)
print("ResidualBlock 입출력:", tuple(feat.shape), "->", tuple(block(feat).shape))
print("기울기가 identity 경로를 타고 그대로 흘러서, 층을 깊게 쌓아도 학습이 된다.")

In [ ]:
# 같은 입력에서 MLP와 CNN의 파라미터 수 비교
print(f"MLP : {PARAM_TABLE['MLP']:>10,} 개")
print(f"CNN : {PARAM_TABLE['CNN']:>10,} 개")
print(f"\nCNN이 약 {PARAM_TABLE['MLP'] / PARAM_TABLE['CNN']:.1f}배 가볍다.")
print("파라미터를 공유하기 때문이고, 그런데도 이미지에서는 CNN이 훨씬 잘 맞는다.")

---

## 3. RNN / LSTM / GRU

순서가 있는 데이터를 앞에서부터 하나씩 읽으면서, 지금까지 읽은 내용을 **은닉 상태(hidden state)** 하나에 압축해 들고 가는 구조야.

$$h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$$

### 바닐라 RNN의 문제

시퀀스가 길어지면 기울기가 역전파되며 계속 곱해져서 0으로 수렴하거나(기울기 소실) 발산해(기울기 폭발). 결국 20~30 스텝 전의 정보는 거의 전달되지 않아.

### LSTM과 GRU

**LSTM**(1997)은 셀 상태(cell state)라는 별도의 고속도로를 만들고, 입력·망각·출력 **게이트**로 정보를 얼마나 흘릴지 조절해. 게이트가 1에 가까우면 정보가 그대로 지나가서 멀리까지 전달돼. ResNet의 스킵 커넥션과 발상이 같아.

**GRU**는 LSTM을 간소화해서 게이트를 2개로 줄인 버전이야. 파라미터가 25% 정도 적고 학습이 빠른데, 성능은 보통 비슷해.

### 요즘은 어떤가

NLP에서는 Transformer에 거의 완전히 밀려났어. 이유는 성능보다 **병렬화**야. RNN은 $h_{t-1}$이 나와야 $h_t$를 계산할 수 있어서 시간축으로 병렬 처리가 불가능해. GPU를 제대로 못 쓰는 구조인 거지.

그래도 살아남은 자리가 있어. 센서 데이터처럼 시퀀스가 짧고 데이터가 적은 시계열, 그리고 메모리가 빠듯한 엣지 디바이스. Transformer는 길이 $n$에 대해 $O(n^2)$ 메모리를 쓰는데 LSTM은 $O(n)$이거든.

### 쓸 때
짧은 시계열 예측, 센서 신호, 소규모 데이터. 길이가 길거나 데이터가 충분하면 Transformer로 가.

In [ ]:
class SimpleLSTM(nn.Module):
    def __init__(self, in_dim=16, hidden=64, num_layers=2, num_classes=5):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=in_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,        # (B, T, F) 순서로 받겠다는 뜻
            bidirectional=False,     # True로 하면 양방향, 파라미터 2배
        )
        self.fc = nn.Linear(hidden, num_classes)

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)   # out: 모든 시점의 출력, h_n: 마지막 은닉 상태
        return self.fc(out[:, -1, :])    # 마지막 시점만 써서 분류


x_seq = torch.randn(8, 20, 16)   # 배치 8, 길이 20, 피처 16
lstm = SimpleLSTM()
show("LSTM", lstm, x_seq)
PARAM_TABLE["LSTM"] = count_params(lstm)

gru = nn.GRU(16, 64, 2, batch_first=True)
print(f"\n같은 설정 GRU 파라미터: {count_params(gru):,} 개 "
      f"(LSTM의 LSTM층만 따로 세면 {count_params(lstm.lstm):,} 개)")

---

## 4. Transformer

2017년 「Attention Is All You Need」. 지금 딥러닝의 기본 골격이야.

### 핵심: Self-Attention

RNN이 "앞에서부터 순서대로 읽는다"면, Attention은 **모든 위치가 모든 위치를 한 번에 본다.** 각 토큰이 Query·Key·Value 세 벡터를 만들고, Query와 Key의 내적으로 "누구를 얼마나 참고할지" 가중치를 계산한 뒤 Value를 가중합해.

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

$\sqrt{d_k}$로 나누는 건 차원이 커질수록 내적 값이 커져서 softmax가 한쪽으로 쏠리는 걸 막기 위해서야.

### 왜 이겼나

- **병렬화** — 모든 위치를 동시에 계산해서 GPU를 꽉 채워 쓸 수 있어. RNN이 못 하던 것
- **거리 무관** — 1번 토큰과 500번 토큰이 한 단계 만에 연결돼. RNN은 500번을 거쳐야 함
- **스케일링** — 데이터와 파라미터를 키우면 성능이 예측 가능하게 올라가. 이 성질이 지금의 대형 모델 시대를 열었어

### 약점

길이 $n$에 대해 attention 행렬이 $n \times n$이라 연산·메모리가 $O(n^2)$야. 긴 문서나 고해상도 영상에서 바로 벽에 부딪혀. 이 비용을 줄이는 게 지난 몇 년간 연구의 큰 줄기였고, 10절의 트렌드 상당수가 여기서 나와.

### 위치 정보

Attention 자체는 순서를 몰라. "나는 밥을 먹었다"와 "밥을 나는 먹었다"가 같아 보여. 그래서 위치 인코딩을 따로 더해줘야 해. 초기에는 사인·코사인 함수를 썼고, 요즘 LLM은 대부분 **RoPE(회전 위치 임베딩)** 를 써 — 긴 문맥으로 확장하기 좋아서.

In [ ]:
class SelfAttention(nn.Module):
    '''Multi-Head Self-Attention을 직접 구현해본 것.'''
    def __init__(self, d_model=64, n_heads=4):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape

        # (B, T, D) -> (B, n_heads, T, d_k) : 헤드별로 쪼개서 병렬 계산
        def split(t):
            return t.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        q, k, v = split(self.q_proj(x)), split(self.k_proj(x)), split(self.v_proj(x))

        # 1) Q·K^T 로 토큰 간 유사도 -> (B, n_heads, T, T)
        scores = (q @ k.transpose(-2, -1)) / (self.d_k ** 0.5)

        # 2) softmax로 "누구를 얼마나 볼지" 가중치
        attn = scores.softmax(dim=-1)

        # 3) Value를 가중합
        out = attn @ v                                   # (B, n_heads, T, d_k)

        # 헤드 합치기
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.out_proj(out), attn


x_tok = torch.randn(2, 10, 64)       # 배치 2, 토큰 10개, 차원 64
attn_layer = SelfAttention()
out, attn_w = attn_layer(x_tok)
print("출력        :", tuple(out.shape))
print("어텐션 행렬 :", tuple(attn_w.shape), "<- (배치, 헤드, 토큰, 토큰)")
print("각 행의 합  :", attn_w[0, 0, 0].sum().item(), "(softmax라 1)")

In [ ]:
# 실무에서는 직접 짜지 않고 PyTorch 내장 모듈을 쓴다
class TinyTransformer(nn.Module):
    def __init__(self, vocab=1000, d_model=64, n_heads=4, n_layers=2, max_len=50, num_classes=5):
        super().__init__()
        self.embed = nn.Embedding(vocab, d_model)
        self.pos = nn.Parameter(torch.zeros(1, max_len, d_model))   # 학습되는 위치 임베딩

        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4,     # 보통 4배로 잡는다
            dropout=0.1, batch_first=True, norm_first=True,  # Pre-LN이 학습이 더 안정적
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, ids):
        T = ids.size(1)
        h = self.embed(ids) + self.pos[:, :T]
        h = self.encoder(h)
        return self.fc(h.mean(dim=1))        # 토큰 평균 풀링 후 분류


ids = torch.randint(0, 1000, (8, 20))
tf = TinyTransformer()
show("Transformer Encoder", tf, ids)
PARAM_TABLE["Transformer"] = count_params(tf)

---

## 5. ViT (Vision Transformer)

2020년. 이미지를 16×16 같은 **패치로 잘라서 각 패치를 하나의 토큰처럼** 취급하고, 그대로 Transformer에 넣어. 합성곱을 쓰지 않고도 이미지 분류 최고 성능을 찍으면서 비전 연구의 방향을 바꿨어.

### CNN과의 차이

CNN은 "가까운 픽셀끼리 관련 있다"는 가정(귀납적 편향, inductive bias)을 구조에 박아둔 모델이야. ViT는 그 가정이 없어. 대신 데이터에서 직접 배워.

이게 장점이자 단점이야. 가정이 없으니 데이터가 충분하면 CNN보다 더 좋은 표현을 학습하는데, 데이터가 적으면 CNN이 공짜로 갖고 있는 것조차 못 배워서 크게 뒤처져.

| | CNN | ViT |
|---|---|---|
| 필요 데이터 | 수천~수만 장부터 쓸만함 | 수백만 장 이상에서 진가 |
| 귀납적 편향 | 지역성·이동 불변성 내장 | 거의 없음 |
| 전역 관계 | 깊이 쌓아야 가능 | 첫 층부터 가능 |
| 해상도 비용 | 선형에 가까움 | 패치 수의 제곱 |
| 소규모 과제 | 유리 | 사전학습 가중치 없으면 불리 |

### 실무에서의 결론

**처음부터 학습(from scratch)할 일은 거의 없어.** ImageNet이나 더 큰 데이터로 사전학습된 가중치를 가져와서 내 데이터에 맞게 미세조정(fine-tuning)하는 게 표준이야. 그러면 "데이터가 적어서 ViT가 불리하다"는 문제가 상당 부분 사라져.

수천 장 규모라면 ResNet이나 EfficientNet 사전학습 모델이 무난하고, 수만 장 이상에 자원이 있으면 ViT 계열이 더 올라가.

In [ ]:
class PatchEmbed(nn.Module):
    '''이미지를 패치로 잘라 토큰 시퀀스로 바꾼다. ViT의 입구.'''
    def __init__(self, img_size=32, patch=4, in_ch=3, d_model=64):
        super().__init__()
        self.n_patches = (img_size // patch) ** 2
        # stride=patch인 conv 한 방이면 "자르기 + 선형변환"이 동시에 된다
        self.proj = nn.Conv2d(in_ch, d_model, kernel_size=patch, stride=patch)

    def forward(self, x):
        x = self.proj(x)              # (B, 3, 32, 32) -> (B, d_model, 8, 8)
        return x.flatten(2).transpose(1, 2)   # -> (B, 64, d_model)


class TinyViT(nn.Module):
    def __init__(self, img_size=32, patch=4, d_model=64, n_heads=4, n_layers=2, num_classes=10):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch, 3, d_model)
        n = self.patch_embed.n_patches

        # CLS 토큰: 이미지 전체를 대표하는 자리. 맨 앞에 붙인다
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos = nn.Parameter(torch.zeros(1, n + 1, d_model))

        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=0.1, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        B = x.size(0)
        h = self.patch_embed(x)
        cls = self.cls_token.expand(B, -1, -1)
        h = torch.cat([cls, h], dim=1) + self.pos
        h = self.encoder(h)
        return self.fc(self.norm(h[:, 0]))    # CLS 토큰 자리만 뽑아서 분류


vit = TinyViT()
print("패치 개수:", vit.patch_embed.n_patches, "(32x32 이미지를 4x4로 자르면 8x8=64개)")
show("ViT", vit, x_img)
PARAM_TABLE["ViT"] = count_params(vit)

---

## 6. 생성 모델

지금까지는 "입력 → 정답"을 맞히는 판별 모델이었어. 생성 모델은 **데이터의 분포 자체**를 배워서 새 샘플을 만들어내.

### AutoEncoder (AE)

입력을 좁은 병목(latent)으로 압축했다가 복원하도록 학습해. 복원이 되려면 중요한 정보만 남겨야 하니까, 병목 벡터가 압축 표현이 돼. 차원 축소나 이상 탐지에 써. 생성용으로는 약해 — latent 공간에 구조가 없어서 아무 벡터나 넣으면 의미 없는 출력이 나와.

### VAE (Variational AutoEncoder, 2013)

latent를 하나의 점이 아니라 **확률 분포**로 만들어. 정규분포에 가깝게 정규화하니까 latent 공간이 연속적이 되고, 거기서 샘플링하면 그럴듯한 새 데이터가 나와. 안정적으로 학습되는 대신 출력이 흐릿한 경향이 있어.

### GAN (2014)

생성자와 판별자를 서로 겨루게 해. 판별자는 진짜/가짜를 구별하려 하고, 생성자는 판별자를 속이려 해. 선명한 결과를 내지만 학습이 불안정하기로 악명 높아 — 모드 붕괴, 발산, 하이퍼파라미터 민감성.

### Diffusion (2020~)

데이터에 노이즈를 조금씩 더해 완전한 잡음으로 만드는 과정을 정의하고, 모델은 그 **역과정(노이즈 제거)** 을 배워. 학습이 GAN보다 훨씬 안정적이고 품질도 높아서 이미지 생성의 주류가 됐어. 단점은 느림 — 샘플 하나를 얻는 데 수십~수백 번 반복해야 해.

### Flow Matching / Rectified Flow (2023~)

Diffusion의 뒤를 잇는 흐름이야. 잡음에서 데이터로 가는 경로를 구불구불한 확률 과정 대신 **거의 직선인 벡터장**으로 학습해. 경로가 곧으니 적은 스텝으로도 좋은 샘플이 나와서 추론이 훨씬 빨라. Stable Diffusion 3, FLUX 같은 최신 이미지·영상 모델이 이 방식으로 넘어갔어.

### 정리

| 모델 | 품질 | 학습 안정성 | 생성 속도 | 지금 위치 |
|---|---|---|---|---|
| AE | — | 매우 안정 | 빠름 | 압축·이상탐지용 |
| VAE | 중 | 안정 | 빠름 | 단독보다 구성 요소로 |
| GAN | 상 | 불안정 | 매우 빠름 | 실시간·경량 용도에 잔존 |
| Diffusion | 최상 | 안정 | 느림 | 여전히 널리 쓰임 |
| **Flow Matching** | 최상 | 안정 | 빠름 | **최신 모델의 기본값으로 이동 중** |

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, in_dim=784, latent=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(),
            nn.Linear(256, 64), nn.ReLU(),
            nn.Linear(64, latent),           # ← 병목
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent, 64), nn.ReLU(),
            nn.Linear(64, 256), nn.ReLU(),
            nn.Linear(256, in_dim), nn.Sigmoid(),   # 0~1 픽셀값으로 복원
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


x_flat = torch.rand(8, 784)        # 28x28 이미지를 평탄화한 셈
ae = AutoEncoder()
show("AutoEncoder", ae, x_flat)
PARAM_TABLE["AutoEncoder"] = count_params(ae)

z = ae.encoder(x_flat)
print(f"\n압축: {tuple(x_flat.shape)} -> {tuple(z.shape)}  "
      f"({784 / z.shape[1]:.0f}배 압축)")
print("손실은 입력과 복원 결과의 차이:  loss = F.mse_loss(ae(x), x)")

In [ ]:
# Diffusion의 순방향(노이즈 추가) 과정 — 개념만 보는 코드
def add_noise(x0, t, T=1000):
    '''t 스텝만큼 노이즈를 섞는다. t가 커질수록 원본이 사라진다.'''
    alpha_bar = torch.cos(torch.tensor(t / T * 3.14159 / 2)) ** 2   # 코사인 스케줄
    noise = torch.randn_like(x0)
    return alpha_bar.sqrt() * x0 + (1 - alpha_bar).sqrt() * noise


x0 = torch.rand(1, 784)
for t in [0, 200, 500, 800, 1000]:
    xt = add_noise(x0, t)
    print(f"t={t:>4}  원본과의 상관계수 = "
          f"{torch.corrcoef(torch.stack([x0.flatten(), xt.flatten()]))[0, 1]:.3f}")

print("\n모델이 배우는 건 이 표의 아래에서 위로 거슬러 올라가는 방법이다.")

---

## 7. GNN (Graph Neural Network)

노드와 엣지로 이뤄진 그래프가 입력일 때 쓰는 모델이야. 소셜 네트워크, 분자 구조, 추천 시스템의 사용자-아이템 관계, 도로망 같은 것들.

핵심 연산은 **메시지 패싱**이야. 각 노드가 이웃들의 정보를 모아서(집계) 자기 표현을 갱신해. 이걸 $k$번 반복하면 $k$홉 떨어진 이웃까지의 정보가 들어와.

$$h_v^{(k+1)} = \text{UPDATE}\left(h_v^{(k)},\ \text{AGGREGATE}\left(\{h_u^{(k)} : u \in N(v)\}\right)\right)$$

집계 방식에 따라 이름이 갈려. **GCN**은 이웃 평균, **GraphSAGE**는 샘플링 후 집계, **GAT**는 어텐션으로 이웃별 가중치를 학습해.

눈치챘겠지만 GAT는 Transformer와 거의 같은 아이디어야. 실제로 Transformer는 "모든 노드가 서로 연결된 완전 그래프 위의 GNN"으로 볼 수 있어.

### 쓸 때
데이터가 표나 이미지가 아니라 **관계 그 자체**일 때. 실무에서는 PyTorch Geometric이나 DGL을 쓰고, 직접 구현할 일은 드물어.

In [ ]:
# 메시지 패싱 한 스텝을 직접 — GCN의 단순 버전
class SimpleGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, h, adj):
        # adj: (N, N) 인접 행렬, h: (N, in_dim) 노드 특징
        deg = adj.sum(dim=1, keepdim=True).clamp(min=1)   # 각 노드의 이웃 수
        agg = (adj @ h) / deg                             # 이웃 특징의 평균 = 메시지 집계
        return F.relu(self.linear(agg))


# 노드 5개짜리 작은 그래프
N = 5
adj = torch.tensor([
    [1, 1, 0, 0, 0],
    [1, 1, 1, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 1, 1, 1],
    [0, 0, 0, 1, 1],
], dtype=torch.float)          # 대각선 1 = 자기 자신도 포함(self-loop)

h = torch.randn(N, 8)
gcn = SimpleGCNLayer(8, 16)
h_new = gcn(h, adj)

print("노드 특징:", tuple(h.shape), "->", tuple(h_new.shape))
print("각 노드의 이웃 수:", adj.sum(dim=1).tolist())
print("한 번 돌리면 1홉, 두 번 돌리면 2홉 이웃의 정보가 섞인다.")

---

## 8. 상황별 선택 가이드

### 먼저 묻는 것

```
정형 데이터(CSV, 표)인가?
├─ 예 → XGBoost / LightGBM 먼저. 딥러닝은 그다음
└─ 아니오 ↓

이미지인가?
├─ 예 → 데이터 몇 장?
│        ├─ ~1만 장  → ResNet/EfficientNet 사전학습 + 미세조정
│        ├─ 1만~10만 → EfficientNet, ConvNeXt
│        └─ 10만 장~ → ViT 계열, DINOv3 같은 자기지도 백본
└─ 아니오 ↓

순서가 있는 데이터인가?
├─ 텍스트  → 사전학습 언어모델 미세조정. 직접 학습은 거의 안 함
├─ 긴 시계열 → Transformer 계열
└─ 짧은 시계열·센서 → LSTM/GRU, 또는 1D CNN
   (데이터가 아주 적으면 고전 통계 모델이 이기는 경우도 많음)

그래프인가? → GNN (PyTorch Geometric)
생성이 목적인가? → Diffusion / Flow Matching
```

### 과제별 정리

| 과제 | 1순위 | 메모 |
|---|---|---|
| 이미지 분류 | ResNet / EfficientNet 사전학습 | 소규모면 거의 항상 전이학습 |
| 객체 검출 | YOLO 계열 | 실시간이면 사실상 표준 |
| 이미지 분할 | U-Net, SAM | 의료 영상은 U-Net이 여전히 강함 |
| 텍스트 분류 | 사전학습 LM 미세조정 | 한국어는 KoBERT·KLUE 계열 |
| 텍스트 생성·요약 | LLM + 프롬프트 또는 LoRA | 전체 미세조정은 비용이 큼 |
| 시계열 예측 | LSTM/GRU, 1D CNN | 단기·소규모면 ARIMA·Prophet도 후보 |
| 이상 탐지 | AutoEncoder, Isolation Forest | 복원 오차를 점수로 |
| 추천 | 행렬 분해, GNN, 투타워 | 데이터 규모가 선택을 가름 |
| 이미지 생성 | Diffusion / Flow Matching | 직접 학습보다 기존 모델 활용 |

### 실무 규칙 세 가지

**가장 단순한 것부터 시작해.** 베이스라인 없이 복잡한 모델을 돌리면 그게 잘한 건지 알 방법이 없어.

**모델보다 데이터를 먼저 봐.** 아키텍처를 바꿔서 얻는 1~2%보다, 라벨 오류를 잡고 증강을 제대로 넣어서 얻는 게 보통 더 커.

**사전학습 가중치를 기본값으로 생각해.** 처음부터 학습하는 건 데이터가 아주 많거나 도메인이 완전히 특수할 때뿐이야.

---

## 9. 파라미터 수 비교

지금까지 만든 모델들을 한눈에. 과제가 제각각이라 직접 비교는 아니지만, 아키텍처별로 파라미터가 어디에 몰리는지 감을 잡기엔 충분해.

In [ ]:
names = list(PARAM_TABLE.keys())
values = [PARAM_TABLE[n] for n in names]

order = np.argsort(values)
names = [names[i] for i in order]
values = [values[i] for i in order]

fig, ax = plt.subplots(figsize=(8, 0.52 * len(names) + 1.8))

bars = ax.barh(names, values, height=0.45, color="#3F6FA8")

# 막대 끝에 값 직접 표기 (고정 간격으로 띄워서 겹치지 않게)
gap = max(values) * 0.015
for bar, v in zip(bars, values):
    ax.text(bar.get_width() + gap, bar.get_y() + bar.get_height() / 2,
            f"{v:,}", va="center", ha="left", fontsize=10, color="#44474C")

ax.set_title("이 노트북에서 만든 모델별 파라미터 수", fontsize=13, pad=14, loc="left")
ax.set_xlabel("학습 가능한 파라미터 (개)", fontsize=10, color="#44474C")
ax.set_xlim(0, max(values) * 1.25)

# 지수 표기(1e6) 대신 천 단위 구분
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))

# 축과 그리드는 뒤로 물러나게
ax.xaxis.grid(True, color="#E8EAED", linewidth=0.8)
ax.set_axisbelow(True)
ax.yaxis.grid(False)
for side in ("top", "right", "left", "bottom"):
    ax.spines[side].set_visible(False)
ax.tick_params(length=0, colors="#44474C", labelsize=10)

plt.tight_layout()
plt.show()

print("\n참고 — 실제 모델들의 규모")
for n, v in [("ResNet-50", 25_600_000), ("EfficientNet-B0", 5_300_000),
             ("ViT-Base", 86_000_000), ("BERT-Base", 110_000_000)]:
    print(f"  {n:<18} {v:>13,} 개")

---

## 10. 2026년 현재 흐름

### 1) Transformer는 여전히 중심, 다만 내부가 바뀌고 있다

구조 자체는 2017년 그대로인데 세부 부품이 거의 다 교체됐어.

- **MoE (Mixture of Experts)** — 전문가 네트워크를 여러 개 두고, 토큰마다 라우터가 그중 2~8개만 켜. 전체 파라미터는 크게 가져가면서 실제 연산은 작은 모델 수준으로 유지하는 방식이야. 대형 모델 쪽에서는 사실상 기본 설계가 됐어.
- **어텐션 효율화** — FlashAttention 계열 커널로 메모리 접근을 줄이고, GQA/MLA로 KV 캐시를 압축하고, 슬라이딩 윈도우로 먼 거리를 솎아내. 다 $O(n^2)$ 비용을 깎는 작업이야.
- **RoPE와 긴 문맥** — 위치 인코딩을 회전 방식으로 바꿔서 백만 토큰 단위 문맥까지 확장하고 있어.

### 2) Post-Transformer — SSM과 하이브리드

**Mamba** 같은 상태공간 모델(SSM)은 시퀀스 길이에 대해 선형으로 스케일해. Transformer의 제곱 비용을 구조적으로 피하는 거지.

다만 순수 SSM이 Transformer를 전면 대체하지는 못했어. 요즘 실제로 쓰이는 건 **하이브리드** — 대부분의 층은 SSM으로 싸게 처리하고, 몇 개 층만 어텐션을 넣어 정확한 참조 능력을 확보하는 식이야.

### 3) 비전: 자기지도 백본으로

라벨 없이 학습한 범용 백본을 **얼려두고(frozen)** 쓰는 방식이 자리를 잡았어. Meta의 **DINOv3**(2025년 8월)가 대표적인데, ViT-7B를 17억 장의 이미지로 라벨 없이 학습했어. 백본을 미세조정하지 않고 위에 선형 레이어만 얹어도 검출·분할 같은 과제에서 전용 모델을 넘어서.

작은 ViT-B/L과 ConvNeXt 버전으로 증류해 배포한다는 점도 눈여겨볼 만해 — **큰 모델로 학습하고 작은 모델로 서빙**하는 지금의 일반적인 패턴이야.

### 4) CNN은 죽지 않았다

ViT가 주목받지만 현장에서 CNN은 여전히 1순위인 경우가 많아. 데이터가 수천~수만 장 규모면 CNN이 더 잘 맞고, 엣지 디바이스에서는 추론 비용이 결정적이거든. ConvNeXt처럼 Transformer의 설계를 가져와 현대화한 CNN도 계속 나오고 있어.

### 5) 생성: Diffusion에서 Flow Matching으로

잡음에서 데이터까지의 경로를 직선에 가깝게 만드는 Flow Matching / Rectified Flow가 확산 모델의 자리를 빠르게 가져가고 있어. 품질은 유지하면서 샘플링 스텝이 크게 줄어서, 영상 생성처럼 비용이 큰 과제에서 특히 의미가 커.

### 6) 작게, 그리고 기기 안에서

파라미터를 무작정 키우는 경쟁은 한풀 꺾였어. 양자화(INT8/INT4), 증류, LoRA 같은 경량 미세조정으로 **작은 모델을 잘 만드는 쪽**에 무게가 실리고 있어. 개인정보와 지연시간 때문에 온디바이스 추론 수요가 계속 늘고 있고.

### 7) 멀티모달이 기본값

텍스트 전용 모델이 오히려 특수한 경우가 됐어. 이미지·오디오·영상을 한 모델이 같이 처리하는 쪽으로 정리되는 중이야.

---

### 공부하는 입장에서 뽑아갈 것

유행하는 아키텍처 이름을 외우는 것보다, **왜 그 구조가 나왔는지**를 잡는 게 오래 가. 스킵 커넥션은 기울기 소실 때문에, 어텐션은 병렬화와 장거리 의존성 때문에, MoE는 연산량 때문에, SSM은 제곱 비용 때문에 나왔어. 문제가 먼저고 구조는 그에 대한 답이야.

그리고 지금 당장 실무에서 가장 자주 쓰게 될 기술은 새 아키텍처 설계가 아니라 **전이학습**이야. 잘 만들어진 사전학습 모델을 가져와 내 데이터에 맞추는 것. 이게 되면 대부분의 과제는 해결돼.

---

## 참고자료

**논문**
- Attention Is All You Need (2017) — Transformer
- Deep Residual Learning for Image Recognition (2015) — ResNet
- An Image is Worth 16x16 Words (2020) — ViT
- Denoising Diffusion Probabilistic Models (2020) — Diffusion
- Mamba: Linear-Time Sequence Modeling with Selective State Spaces (2023)
- DINOv3 (2025) — 자기지도 비전 백본

**문서**
- PyTorch 공식 튜토리얼: https://pytorch.org/tutorials/
- torchvision 사전학습 모델: https://pytorch.org/vision/stable/models.html
- Hugging Face: https://huggingface.co/docs
- Papers with Code (과제별 SOTA): https://paperswithcode.com/

**다음에 해볼 것**
1. torchvision에서 ResNet18 사전학습 모델을 받아 Intel Image Set에 미세조정 — 처음부터 학습한 CNN과 정확도 비교
2. 같은 데이터에 ViT를 붙여보고 데이터 양에 따른 차이 확인
3. 마지막 conv 층의 특징 맵을 시각화해서 모델이 어디를 보는지 확인 (Grad-CAM)